In [44]:
import matplotlib.pyplot as plt
import read_timeseries as rd 
from pathlib import Path
import os
import re
import pandas as pd


import matplotlib.dates as mdates
from datetime import datetime
import numpy as np
import pandas as pd

### functions

In [ ]:
# functions for reading
def read_files(folder, name, base_dir):
    """
    Read all `.srh` and `.srw` files in `base_dir / folder` whose names are
    `name_1`, `name_2`, ... For each index, columns from the `.srw` file are
    appended to the `.srh` file.

    Parameters
    ----------
    folder : str
        Subfolder inside `base_dir` (e.g. 'Run1').
    name : str
        Base file name without the `_<index>` suffix (e.g. '36_17').
    base_dir : Path

    Returns
    -------
    dict
        Mapping from the suffix index (int) to the merged DataFrame.
    """
    folder_path = Path(base_dir) / folder

    if not folder_path.is_dir():
        raise NotADirectoryError(f"Folder does not exist: {folder_path.resolve()}")

    # Match "<name>_<number>.srh" or "<name>_<number>.srw"
    pattern = re.compile(rf"^{re.escape(name)}_(\d+)\.(srh|srw)$", re.IGNORECASE)

    # Collect files grouped by index: {idx: {'srh': path, 'srw': path}}
    grouped = {}
    for entry in folder_path.iterdir():
        if not entry.is_file():
            continue
        m = pattern.match(entry.name)
        if m:
            idx = int(m.group(1))
            ext = m.group(2).lower()
            grouped.setdefault(idx, {})[ext] = entry

    if not grouped:
        raise FileNotFoundError(
            f"No files matching '{name}_<index>.srh/.srw' found in {folder_path.resolve()}"
        )

    results = {}
    for idx in sorted(grouped):
        files = grouped[idx]
        srh_path = files.get('srh')
        srw_path = files.get('srw')

        try:
            if srh_path is not None:
                df_srh = rd.get_mohid_timeseries_1_file(str(srh_path))
            else:
                df_srh = None
                print(f"Warning: no .srh file for index {idx}")

            if srw_path is not None:
                df_srw = rd.get_mohid_timeseries_1_file(str(srw_path))
            else:
                df_srw = None

            if df_srh is not None and df_srw is not None:
                # Drop columns from srw that already exist in srh to avoid duplicates
                new_cols = [c for c in df_srw.columns if c not in df_srh.columns]
                # Align on index (usually the time index) and append the new columns
                merged = df_srh.join(df_srw[new_cols], how='left')
                results[idx] = merged
            elif df_srh is not None:
                results[idx] = df_srh
            elif df_srw is not None:
                results[idx] = df_srw

        except Exception as e:
            print(f"Warning: failed to read files for index {idx}: {e}")

    return results

In [71]:
# functions for statistics
def rmse_vs_reference(run, reference, columns=None):
    """
    Compute RMSE between each DataFrame in `run` and the matching DataFrame
    in `reference`, keyed by index.

    Parameters
    ----------
    run : dict[int, pd.DataFrame]
        Output of `read_files` for the run being evaluated (e.g. Run2).
    reference : dict[int, pd.DataFrame]
        Output of `read_files` for the reference run (e.g. Run1).
    columns : list[str], optional
        Restrict the comparison to these columns. If None, all columns common
        to both DataFrames are used.

    Returns
    -------
    pd.DataFrame
        Rows = suffix index, columns = data columns, values = RMSE.
    """
    rmse_rows = {}

    for idx, df_run in run.items():
        if idx not in reference:
            print(f"Warning: index {idx} not in reference, skipping.")
            continue

        df_ref = reference[idx]

        # Decide which columns to compare
        if columns is None:
            cols = [c for c in df_run.columns if c in df_ref.columns]
        else:
            cols = [c for c in columns
                    if c in df_run.columns and c in df_ref.columns]

        if not cols:
            print(f"Warning: no common columns for index {idx}, skipping.")
            continue

        # Align on the (time) index so rows correspond
        a, b = df_run[cols].align(df_ref[cols], join='inner', axis=0)

        # Column-wise RMSE, ignoring NaNs
        diff = (a - b) ** 2
        rmse_rows[idx] = np.sqrt(diff.mean(axis=0, skipna=True))

    return pd.DataFrame.from_dict(rmse_rows, orient='index').sort_index()

def rmse_all_runs(all_results, reference_key='Run1', columns=None):
    ref = all_results[reference_key]
    return {
        run_key: rmse_vs_reference(run, ref, columns=columns)
        for run_key, run in all_results.items()
        if run_key != reference_key
    }

def sensitivity_index(run, reference, columns=None):
    """
    Compute the Sensitivity Index (SI) between each DataFrame in `run` and the
    matching DataFrame in `reference`, keyed by index.

        SI = RMSE / (Q_ref_max - Q_ref_min)

    where Q_ref_max and Q_ref_min are the max and min of the reference series
    for each column, at each index.

    Parameters
    ----------
    run : dict[int, pd.DataFrame]
    reference : dict[int, pd.DataFrame]
    columns : list[str], optional
        Restrict the comparison to these columns. If None, all common columns
        are used.

    Returns
    -------
    pd.DataFrame
        Rows = suffix index, columns = data columns, values = SI.
    """
    si_rows = {}

    for idx, df_run in run.items():
        if idx not in reference:
            print(f"Warning: index {idx} not in reference, skipping.")
            continue

        df_ref = reference[idx]

        if columns is None:
            cols = [c for c in df_run.columns if c in df_ref.columns]
        else:
            cols = [c for c in columns
                    if c in df_run.columns and c in df_ref.columns]

        if not cols:
            print(f"Warning: no common columns for index {idx}, skipping.")
            continue

        # Align on the (time) index so rows correspond
        a, b = df_run[cols].align(df_ref[cols], join='inner', axis=0)

        # Column-wise RMSE
        rmse = np.sqrt(((a - b) ** 2).mean(axis=0, skipna=True))

        # Reference range per column
        ref_range = b.max(axis=0, skipna=True) - b.min(axis=0, skipna=True)

        # Avoid division by zero -> NaN where reference is constant
        ref_range = ref_range.replace(0, np.nan)

        si_rows[idx] = rmse / ref_range

    return pd.DataFrame.from_dict(si_rows, orient='index').sort_index()

def sensitivity_index_all_runs(all_results, reference_key='Run1', columns=None):
    ref = all_results[reference_key]
    return {
        run_key: sensitivity_index(run, ref, columns=columns)
        for run_key, run in all_results.items()
        if run_key != reference_key
    }

def si_by_layer(all_results, reference_key='Run1', columns=None):
    """
    Reshape sensitivity indices into one DataFrame per layer (suffix index).

    Parameters
    ----------
    all_results : dict[str, dict[int, pd.DataFrame]]
        e.g. {'Run1': {1: df, 2: df, ...}, 'Run2': {...}, ...}
    reference_key : str
        Key in `all_results` to use as the reference run.
    columns : list[str], optional
        Restrict the comparison to these columns.

    Returns
    -------
    dict[int, pd.DataFrame]
        Mapping from layer index -> DataFrame whose rows are runs and whose
        columns are ['Run', 'si_<column1>', 'si_<column2>', ...].
    """
    ref = all_results[reference_key]

    # {layer_idx: list of rows, one per run}
    per_layer_rows = {}

    for run_key, run in all_results.items():
        if run_key == reference_key:
            continue

        si_df = sensitivity_index(run, ref, columns=columns)
        # si_df: rows = layer index, columns = data columns

        for layer_idx, row in si_df.iterrows():
            record = {'Run': run_key}
            record.update({f'si_{col}': row[col] for col in si_df.columns})
            per_layer_rows.setdefault(layer_idx, []).append(record)

    return {
        layer_idx: pd.DataFrame(rows).reset_index(drop=True)
        for layer_idx, rows in per_layer_rows.items()
    }

### reading files

In [68]:
base_dir = Path(r'C:\Users\karoa\MOHID_internship\MOHID_single_domain\res')

folders = ['Run1', 'Run2', 'Run3']
name = '36_17' 

all_results = {folder: read_files(folder, name, base_dir) for folder in folders}


### calculate statistics

In [70]:
rmses = rmse_all_runs(all_results, reference_key='Run1')
sis = sensitivity_index_all_runs(all_results, reference_key='Run1')
sis['Run2']   # DataFrame of SIs for Run2 vs Run1

,velocity_U,velocity_V,velocity_W,velocity_modulus,velocity_direction,water_level,OpenPoint,salinity,temperature
1,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
10,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0


In [ ]:
si_layers = si_by_layer(all_results, reference_key='Run1')

si_layers[13]



,Run,si_velocity_U,si_velocity_V,si_velocity_W,si_velocity_modulus,si_velocity_direction,si_water_level,si_OpenPoint,si_salinity,si_temperature
0,Run2,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0
1,Run3,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0
